Compare two traces for the same vorgang side by side

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import mlflow
from IPython.display import display as ipython_display

from pazufa_scraper_be.tracing.compare import (
    find_trace,
)
from pazufa_scraper_be.tracing.render_pipeline import two_pipeline_trace_inputs_viewer, two_pipeline_trace_outputs_viewer
from pazufa_scraper_be.tracing.trace_pipeline import get_traces_from_mlflow_db

In [ ]:
root_path = Path.cwd().parent

db_path_reference = (root_path / "tests" / "e2e-reference-mlflow.db").resolve()

assert db_path_reference.exists()

In [ ]:
tracking_uri_reference = f"sqlite:///{db_path_reference}"
experiment_reference = "e2e reference"

In [ ]:
span_name = "single-item crawl"

In [ ]:
mlflow.set_tracking_uri(tracking_uri_reference)
mlflow.set_experiment(experiment_reference)

In [ ]:
traces_reference = get_traces_from_mlflow_db(
    span_name=span_name,
    tracking_uri=tracking_uri_reference,
    experiment=experiment_reference,
)
len(traces_reference)

In [ ]:
db_path_test_run = (root_path / "tests" / "e2e-tests-mlflow.db").resolve()
assert db_path_test_run.exists()

In [ ]:
tracking_uri_test_run = f"sqlite:///{db_path_test_run}"
experiment_test_run = "e2e test run"

mlflow.set_tracking_uri(tracking_uri_test_run)
mlflow.set_experiment(experiment_test_run)

In [ ]:
traces_test_run = get_traces_from_mlflow_db(
    span_name=span_name,
    tracking_uri=tracking_uri_test_run,
    experiment=experiment_test_run,
)
len(traces_test_run)

In [ ]:
comparison_span_name = "BuildPaZuFaVorgang"
comparison_item_id = "V-351529"

In [ ]:
trace_reference = find_trace(traces_reference, comparison_item_id)

trace_reference

In [ ]:
trace_test_run = find_trace(traces_test_run, comparison_item_id)
trace_test_run

In [ ]:
if trace_reference is None:
    msg = f"Reference trace not found for item {comparison_item_id}"
    raise LookupError(msg)
if trace_test_run is None:
    msg = f"Test-run trace not found for item {comparison_item_id}"
    raise LookupError(msg)

ipython_display(two_pipeline_trace_inputs_viewer(trace_reference, trace_test_run, span_name=span_name))

In [ ]:
ipython_display(two_pipeline_trace_outputs_viewer(trace_reference, trace_test_run, span_name=span_name))